In [3]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

In [5]:
import pandas as pd
import numpy as np

## Data cleaning
Kept the last 15years of data

In [9]:
input_file = "data/SP500_Historical_Data.csv"
output_file = "data/SP500_Historical_Data.csv"

df = pd.read_csv(input_file)

df["Date"] = pd.to_datetime(df["Date"])

cutoff = pd.Timestamp.now() - pd.DateOffset(years=10)

df_small = df[df["Date"] >= cutoff]

df_small.to_csv(output_file, index=False)

print(f"Rows kept: {len(df_small)}")

Rows kept: 1120764


In [ ]:
def load_ticker(path: str, ticker: str) -> pd.Series:
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker.upper()].sort_values("Date")
    if df.empty:
        raise ValueError(f"Ticker '{ticker}' no encontrado en el dataset.")
    series = df.set_index("Date")["Close"].dropna()
    print(f"[load] {ticker} → {len(series)} filas | {series.index[0].date()} a {series.index[-1].date()}")
    return series

# Main

In [ ]:
path = "data/SP500_Historical_Data.csv"
execute_predictions(path, "AAPL")

[load] AAPL → 2445 filas | 2016-06-01 a 2026-02-20

  Filtro de Kalman (1D)
  Proceso σ²     : 1e-05
  Medición σ²    : 0.1
  Precio raw     : 264.5800
  Precio filtrado: 248.8569
  Ruido estimado : +15.7231
  Última ganancia K: 0.009950

  Últimos 5 días (raw vs filtrado):
               Raw      Kalman
Date                          
2026-02-13  255.78  248.265564
2026-02-17  263.88  248.420930
2026-02-18  264.35  248.579426
2026-02-19  260.58  248.698833
2026-02-20  264.58  248.856853


In [ ]:
def execute_predictions(path, name):
    price  = load_ticker(path, name)
    kalman(price)

# Kalman Filter

In [ ]:
def extract_stock(input_csv, ticker, output_csv=None):
    df = pd.read_csv(input_csv)

    stock_df = df[df["Ticker"] == ticker].copy()

    if output_csv:
        stock_df.to_csv(output_csv, index=False)

    return stock_df

In [ ]:
aapl = extract_stock(
    "data/SP500_Historical_Data.csv",
    "AAPL",
    "processed_data/AAPL.csv"
)

In [ ]:
def kalman(prices: pd.Series, process_var: float = 1e-5, measurement_var: float = 0.1) -> str:
    n = len(prices)
    vals = prices.values
    x_est = np.zeros(n)   # estimación del estado
    p_est = np.zeros(n)   # estimación del error
 
    x_est[0] = vals[0]
    p_est[0] = 1.0
 
    for i in range(1, n):
        x_pred = x_est[i - 1]
        p_pred = p_est[i - 1] + process_var
 
        K = p_pred / (p_pred + measurement_var)
 
        x_est[i] = x_pred + K * (vals[i] - x_pred)
        p_est[i] = (1 - K) * p_pred
 
    smoothed = pd.Series(x_est, index=prices.index, name="Kalman")
 
    last_raw  = vals[-1]
    last_smoothed = x_est[-1]
    last_gain = K  # última ganancia
    noise = last_raw - last_smoothed
 
    lines = [
        f"\n{'='*50}",
        f"  Filtro de Kalman (1D)",
        f"{'='*50}",
        f"  Proceso σ²     : {process_var}",
        f"  Medición σ²    : {measurement_var}",
        f"  Precio raw     : {last_raw:.4f}",
        f"  Precio filtrado: {last_smoothed:.4f}",
        f"  Ruido estimado : {noise:+.4f}",
        f"  Última ganancia K: {last_gain:.6f}",
        f"\n  Últimos 5 días (raw vs filtrado):",
    ]
    tail = pd.DataFrame({"Raw": prices, "Kalman": smoothed}).tail(5)
    lines.append(tail.to_string())
    print("\n".join(lines))
    return "\n".join(lines)
